## 1. Importing Necessary Libraries
This initial cell imports all the required Python libraries for the script. This includes os for file system operations, scipy for loading data from MATLAB's .mat files, matplotlib for plotting, numpy for numerical operations, and several key components from the pynwb library for creating and managing the Neurodata Without Borders (NWB) file.

### Attention:
Bear in mind that once nwb objects (unit data or behavior) are defined they cannot be modified and will cause errors if attempting to re-run the cell. Thus I advice restarting the kernel and running the notebook from the top,

In [1]:
import os
import scipy
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb.ecephys import ElectrodeGroup

## 2. Defining File Paths and Session Names
Useful if needed to process all sessions at once. 

In [2]:
directory = r"\\research-cifs.nyumc.org\research\buzsakilab\Homes\voerom01\TES\TES_superResponders"

names_w_tracking_ripples_odors = ['TES_sResp_M01_20240308_odor','TES_sResp_M02_20240307_odor','TES_sResp_M02_20240226_odor','TES_sResp_M05_20240727']

## 3. Loading, Processing, and Organizing Data
This cell will load all data using cell explorer format and organize the variables we want to export as nwb

In [3]:
name = names_w_tracking_ripples_odors[3]

print(name)

os.chdir(directory+"\\"+name[0:13]+"\\"+name)

odor_trials = scipy.io.loadmat('Odor_sniff_info.mat',simplify_cells=True)['odorInfo']['odorTrials']
odor_trials_empty = odor_trials[-1]-1

# get cells and ids

cell_info = scipy.io.loadmat(name+'.cell_metrics.cellinfo.mat',simplify_cells=True)['cell_metrics']

cell_types = cell_info['putativeCellType']

# check bad cells

if "tags" in cell_info:
    tags = cell_info["tags"]
    if 'Bad' in list(tags):
        bad_cells = cell_info["tags"]['Bad']
    else:
        bad_cells = []
        
    cell_id = cell_info["cellID"]
    good_cells = np.setdiff1d(cell_id,bad_cells)-1
else:
    good_cells = cell_info["cellID"]-1

cell_types = cell_types[good_cells]

spike_times_all = cell_info["spikes"]["times"][good_cells]
shank_id_all = cell_info['shankID'][good_cells]

if name[10:13] == 'M01':
    
        rsc_shanks = [3,6]
        ca1_shanks = [2,5]
        ca3_shanks = [1,4]
    
    # if name[10:13] == 'M02':
    
    #     rsc_shanks = [3,5]
    #     ca1_shanks = [2,4,6]
    #     ca3_shanks = [1]
    
if name == 'TES_sResp_M02_20240226_odor':

    rsc_shanks = [2,5]
    ca1_shanks = [1,4]
    ca3_shanks = [3]
    
if name == 'TES_sResp_M02_20240307_odor':

    rsc_shanks = [3,6,7]
    ca1_shanks = [2,5]
    ca3_shanks = [1,4]
    
if name[10:13] == 'M03':

    rsc_shanks = [5,6]
    ca1_shanks = [2,3,4]
    ca3_shanks = [1]

if name[10:13] == 'M05':

    rsc_shanks = [4,6,8]
    ca1_shanks = [3,5,7]
    ca3_shanks = [1,2]
    

cell_area = []
for x in range(cell_types.shape[0]):
    if np.isin(shank_id_all[x],ca1_shanks):
        cell_area.append("CA1")
    if np.isin(shank_id_all[x],ca3_shanks):
        cell_area.append("CA3")
    if np.isin(shank_id_all[x],rsc_shanks):
        cell_area.append("RSC")
cell_area = np.array(cell_area)

pyramidal_cells_rsc = (cell_types != 'Narrow Interneuron') * (cell_area == "RSC")
pyramidal_cells_ca1 = (cell_types == 'Pyramidal Cell') * (cell_area == "CA1")
pyramidal_cells_ca3 = (cell_types == 'Pyramidal Cell') * (cell_area == "CA3")

interneurons_ca1 = (cell_types != 'Pyramidal Cell') * (cell_area == "CA1")
interneurons_ca3 = (cell_types != 'Pyramidal Cell') * (cell_area == "CA3")
interneurons_rsc = (cell_types == 'Narrow Interneuron') * (cell_area == "RSC")

os.chdir(r'C:\Users\GONZAJ81\Downloads')
y_chan_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)["ycoords"]
x_chan_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)["xcoords"]
cell_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)
max_ch_amp = cell_info["maxWaveformCh"][good_cells]
y_cell_coordinates = y_chan_coordinates[max_ch_amp]
x_cell_coordinates = x_chan_coordinates[max_ch_amp]

firing_rates = cell_info['firingRate'][good_cells]
ab_ratio = cell_info['ab_ratio'][good_cells]
acg = cell_info['acg']['wide'][good_cells]
acg_t = cell_info['acg']['narrow'].T
acg = acg_t[good_cells]
burstIndex_Mizuseki2012 = cell_info['burstIndex_Mizuseki2012'][good_cells]
cv2 = cell_info['cv2'][good_cells]
maxWaveformCh = cell_info['maxWaveformCh'][good_cells]
troughToPeak = cell_info['troughToPeak'][good_cells]
waveforms = cell_info['waveforms']['raw'][good_cells]
acg_tau_decay = cell_info['acg_tau_decay'][good_cells]
acg_tau_rise = cell_info['acg_tau_rise'][good_cells]
thetaModulationIndex = cell_info['thetaModulationIndex'][good_cells]
y_position_probe = y_cell_coordinates
x_position_probe = x_cell_coordinates

metrics_2_add = [spike_times_all,cell_types,cell_area,firing_rates,ab_ratio,acg,burstIndex_Mizuseki2012,cv2,maxWaveformCh,troughToPeak,waveforms,acg_tau_decay,acg_tau_rise,thetaModulationIndex, x_position_probe, y_position_probe]

TES_sResp_M05_20240727


In [4]:
import mat73
os.chdir(directory+"\\"+name[0:13]+"\\"+name)
session = scipy.io.loadmat(name+'.session.mat', simplify_cells = True)["session"]
odor_data = scipy.io.loadmat(name+'.odorManipulation.manipulation.mat', simplify_cells = True)['odorManipulation']

odor_trials = scipy.io.loadmat('Odor_sniff_info.mat',simplify_cells=True)['odorInfo']['odorTrials']
odor_trials_empty = odor_trials[-1]-1

odor_times_on = odor_data['valve_On_timestamps']
odor_times_off = odor_data['valve_Off_timestamps'][:,1]

#odor_times_off = odor_data['valve_On_timestamps']
#odor_times_on = odor_data['valve_Off_timestamps'][:,1]  
    
if odor_times_on[0]>odor_times_off[0]:
    print('here problem')
    odor_times_off = odor_data['valve_On_timestamps']
    odor_times_on = odor_data['valve_Off_timestamps'][:,1]
    
odor_trials = scipy.io.loadmat('Odor_sniff_info.mat',simplify_cells=True)['odorInfo']['odorTrials']
odor_trials_empty = odor_trials[-1]-1
odor_trials_1 = odor_trials[0]-1
odor_trials_2 = odor_trials[1]-1
odor_trials_3 = odor_trials[2]-1

odor_1_times = np.vstack([odor_times_on[odor_trials_1],odor_times_off[odor_trials_1]]).T
odor_2_times = np.vstack([odor_times_on[odor_trials_2],odor_times_off[odor_trials_2]]).T
odor_3_times = np.vstack([odor_times_on[odor_trials_3],odor_times_off[odor_trials_3]]).T
odor_empty_times = np.vstack([odor_times_on[odor_trials_empty],odor_times_off[odor_trials_empty]]).T


here problem


In [5]:
odor_1_times
odor_times_on[0]>odor_times_off[0]

False

## 4. Creating the NWB File Object
This is the first step in building the NWB file. An NWBFile object is instantiated, serving as the main container for all experimental data and metadata. Essential metadata, such as the session description, start time, experimenter, and lab, are provided here.

In [6]:
# 1. Create NWBFile (Metadata is crucial)
session_start_time = datetime(2024, 7, 27, 10, 0, 0, tzinfo=pytz.utc)
nwbfile = NWBFile(
    session_description='Head Fix Session with Odor Stimulation',
    identifier='name',
    session_start_time=session_start_time,
    experimenter='Mihály Vöröslakos',
    lab='Buzsáki Lab',
    institution='NYU',
    # Add other required fields...
)

# weeks m01 6-15, m02 8-15, m03 6-15, M05 8-15
from pynwb.file import Subject
# 2. Add Subject metadata directly
nwbfile.subject = Subject(
    subject_id='M05',
    species='Mus musculus',
    strain='C57BL/6J',
    sex='F',
    age='P8W/P15W',      # Estimated age range for 21g female
    #weight='0.021 kg',
    description='Wild-type mouse'
)

## 5. Adding Odor Stimulation Intervals
This section handles time-based event data. First, it loads .mat files containing the start and end times for odors. It then creates a TimeIntervals table named OdorStimulus and populates it by adding a row for each detected event, including its start time, stop time, and a descriptive label (e.g., 'NREM', 'Ripple')

In [7]:
from pynwb.epoch import TimeIntervals

# 1. Initialize the Table
odor_stimulus = TimeIntervals(name='Odor Stimulus', 
                             description='Periods of odor delivery.')

# Define the custom column for the state label
odor_stimulus.add_column(name='stimulus', 
                        description='Odor Stim')

# 2. Collect all events into a single list
all_events = []

# Helper function to append data to our list
def collect_events(data_source, label):
    for row in data_source:
        all_events.append({
            'start_time': float(row[0]),
            'stop_time': float(row[1]),
            'state': label
        })

#rem_states = rem_states[np.newaxis,:] #if there is a single interval

# Collect from your different variables
collect_events(odor_1_times, 'Odor 1')
collect_events(odor_1_times, 'Odor 1')
collect_events(odor_1_times, 'Odor 1')
collect_events(odor_empty_times, 'Odor Empty')

# 3. SORT the list by start_time (CRITICAL STEP)
# This ensures DANDI validation passes
all_events.sort(key=lambda x: x['start_time'])

# 4. Add the sorted events to the NWB table
for event in all_events:
    odor_stimulus.add_row(
        start_time=event['start_time'],
        stop_time=event['stop_time'],
        stimulus=event['state']
    )

# 5. Add to the nwbfile
nwbfile.add_time_intervals(odor_stimulus)

print(f"Total sorted events added: {len(all_events)}")

Total sorted events added: 201


## 7. Adding Spike Data and Cell Metrics to the Units Table

This is a critical step where all the spike data and associated cell metrics are added to the NWB file. The process involves three main parts:

Defining Hardware Metadata: An ElectrodeGroup is created to provide context about the recording probe.
Defining the Units Table Structure: Custom columns are added to the NWB file's units table using add_unit_column. Each column is given a name and a description, defining the structure that will hold all the detailed metrics for each neuron.
Populating the Table: The code iterates through each neuron, adding a new row to the units table using add_unit(). Each row is populated with the neuron's spike times and all of its corresponding metrics (cell type, firing rate, waveform shape, etc.).

In [8]:
# 2. Add an Electrodes table (essential for units)
# You would get this info from your cell_metrics or a separate file

#nwbfile.add_electrode_group(
#    name='ElectrodeGroup1',
#    description='45 degree insertion targetting hippocampus and neocortex',
#    location='CA3, CA1, RSC',
#    device=device
#)

device = nwbfile.create_device(name='Neuropixels 2.0')

e_group = ElectrodeGroup(
    name='ElectrodeGroup1',
    description='60 degree insertion targetting hippocampus and neocortex',
    location='CA3, CA1, RSC',
    device=device  # Pass the device object here
)
nwbfile.add_electrode_group(e_group)

# --- ADD THIS SECTION BEFORE THE LOOP ---

# A. Non-Waveform Metrics (simple scalars or 1D arrays)
nwbfile.add_unit_column(name='cell_type', description='Cell classification (e.g., Pyramidal, Narrow Interneuron, Wide Interneuron)')
nwbfile.add_unit_column(name='cell_area', description='The brain region or subfield the cell was assigned to (e.g., CA1, CA3, RSC)')
nwbfile.add_unit_column(name='firing_rate', description='Firing rate in Hz: Spike count normalized by the interval between the first and the last spike..')
nwbfile.add_unit_column(name='ab_ratio', description='Waveform asymmetry; the ratio between the two positive peaks (peakB-peakA)/(peakA+peakB).')
nwbfile.add_unit_column(name='burstIndex_Mizuseki2012', description='Burst index as defined by Mizuseki et al. 2012.')
nwbfile.add_unit_column(name='cv2', description='Coefficient of variation (CV_2, 10.1152/jn.1996.75.5.1806).')
nwbfile.add_unit_column(name='maxWaveformCh', description='Max channel zero-indexed: The channel with the largest amplitude.')
nwbfile.add_unit_column(name='troughToPeak', description='Trough-to-peak latency is defined from the trough to the following peak of the waveform.')
nwbfile.add_unit_column(name='acg_tau_decay', description='Decay constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='acg_tau_rise', description='Rise constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='thetaModulationIndex', description='Theta modulation index. Originally defined in Cacucci et al., JNeuro 2004. Computed as the difference between the theta modulation trough (defined as mean of autocorrelogram bins, 50-70 msec) and the theta modulation peak (mean of autocorrelogram bins, 100-140 msec) over their sum, scaled from -1 to 1.')
nwbfile.add_unit_column(name='x_position_probe', description='Position along the x-axis for the max amp channel for each cell')
nwbfile.add_unit_column(name='y_position_probe', description='Position along the y-axis for the max amp channel for each cell')

# B. Complex Data (Waveforms and ACG)
# These are typically 1D arrays per unit, so we set the dtype to 'object' 
# to allow arrays of variable length/content (like NumPy arrays) to be stored in the column.
#nwbfile.add_unit_column(name='waveforms', description='Average raw spike waveform from channel with max amplitude.', dtype='object')
#nwbfile.add_unit_column(name='acg', description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', dtype='object')

# 1. WAVEFORMS
nwbfile.add_unit_column(
    name='waveforms', 
    description='Average raw spike waveform from channel with max amplitude.', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# 2. ACG
nwbfile.add_unit_column(
    name='acg', 
    description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# --- MODIFIED LOOP SECTION ---
# 3. Add Units table data
# Assuming cell_metrics is the original dict (used for firing_rate)
# and your new metrics are lists/arrays indexed by unit_i

for unit_i in range(len(spike_times_all)):
    # Get spike times (must be a 1D NumPy array in seconds)
    spike_times_list = spike_times_all[unit_i]
    
    nwbfile.add_unit(
        # Required arguments
        spike_times=spike_times_list,
        id=unit_i + 1,  # Unit IDs start at 1
        
        # --- ADDING YOUR METRICS ---
        
        # Scalar Metrics (from your list 'metrics_2_add')
        cell_type=cell_types[unit_i],
        cell_area=cell_area[unit_i],
        # Note: 'firing_rate' already existed in the original example, 
        # so we keep that structure if possible:
        firing_rate=firing_rates[unit_i], 
        ab_ratio=ab_ratio[unit_i],
        burstIndex_Mizuseki2012=burstIndex_Mizuseki2012[unit_i],
        cv2=cv2[unit_i],
        maxWaveformCh=maxWaveformCh[unit_i],
        troughToPeak=troughToPeak[unit_i],
        acg_tau_decay=acg_tau_decay[unit_i],
        acg_tau_rise=acg_tau_rise[unit_i],
        thetaModulationIndex=thetaModulationIndex[unit_i],
        x_position_probe=x_position_probe[unit_i],
        y_position_probe=y_position_probe[unit_i],

        # Array Metrics (WAVEFORMS and ACG)
        waveforms=waveforms[unit_i][:,np.newaxis],  # Must be a 1D or 2D NumPy array
        acg=acg[unit_i][:,np.newaxis] # Must be a 1D NumPy array
    )

C:\Users\GONZAJ81\AppData\Local\anaconda3\Lib\site-packages\pynwb\file.py:719: UserWarning: Column 'waveforms' is predefined in Units with index=2 which does not match the entered index argument. The predefined index spec will be ignored. Please ensure the new column complies with the spec. This will raise an error in a future version of HDMF.
  self.units.add_column(**kwargs)


## 8. Saving all data into the .nwb file

In [9]:
os.chdir(r'C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695')
# 4. Write the file
with NWBHDF5IO(name+'.nwb', 'w') as io:
    io.write(nwbfile)

print("NWB file created successfully!")

NWB file created successfully!


## 9. Validate the .nwb file 
### This is mandatory for DANDI Upload, if needed you should install DANDI packages

In [10]:
filename = name+".nwb"

# Use the $ symbol to inject the variable into the shell command
!dandi validate $filename

[DANDI.NON_DANDI_FILENAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\TES_sResp_M05_20240727.nwb — Filename does not conform to DANDI standard
[DANDI.NON_DANDI_FOLDERNAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\TES_sResp_M05_20240727.nwb — File is not in folder at root with subject name


2026-01-13 22:32:40,309 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 22:32:40,310 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 22:32:42,609 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-03.32.39Z-26752.log


from pynwb.epoch import TimeIntervals

# Assuming you have the following variables loaded:
# sleep_start_times: 1D array of state start times (in seconds)
# sleep_stop_times: 1D array of state end times (in seconds)
# sleep_state_labels: 1D array of strings (e.g., ['WAKE', 'NREM', 'REM', ...])

# Create the TimeIntervals table
sleep_states = TimeIntervals(name='SleepStates', 
                             description='Periods of WAKE, NREM, and REM sleep.')

# Define the custom column for the state label
sleep_states.add_column(name='state', 
                        description='The behavioral state label (WAKE, NREM, or REM).')

# Add the TimeIntervals table to the NWBFile
nwbfile.add_time_intervals(sleep_states)

# --- Add NREM States ---
nrem_states_data = nrem_states  # Your input variable
state_label = 'NREM'

for row in nrem_states_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    sleep_states.add_row(
        start_time=start_time,
        stop_time=stop_time,
        state=state_label
    )

print(f"Successfully added {len(nrem_states_data)} '{state_label}' intervals.")

# --- Add REM States ---
rem_states_data = rem_states  # Your input variable
#rem_states_data = rem_states_data[np.newaxis,:] if there is a single interval
state_label = 'REM'

for row in rem_states_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    sleep_states.add_row(
        start_time=start_time,
        stop_time=stop_time,
        state=state_label
    )

print(f"Successfully added {len(rem_states_data)} '{state_label}' intervals.")

# --- Add Wake States ---
wake_states_data = wake_states  # Your input variable
state_label = 'WAKE'

for row in wake_states_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    sleep_states.add_row(
        start_time=start_time,
        stop_time=stop_time,
        state=state_label
    )

print(f"Successfully added {len(wake_states_data)} '{state_label}' intervals.")

# --- Add ripples States ---
ripple_states_data = ripple_times  # Your input variable
state_label = 'Ripple'

for row in ripple_states_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    sleep_states.add_row(
        start_time=start_time,
        stop_time=stop_time,
        state=state_label
    )

print(f"Successfully added {len(ripple_states_data)} '{state_label}' intervals.")

from pynwb.epoch import TimeIntervals

# Assuming you have the following variables loaded:
# sleep_start_times: 1D array of state start times (in seconds)
# sleep_stop_times: 1D array of state end times (in seconds)
# sleep_state_labels: 1D array of strings (e.g., ['WAKE', 'NREM', 'REM', ...])

# Create the TimeIntervals table
odor_stimulus = TimeIntervals(name='Odor Stimulus', 
                             description='Periods of odor delivery.')

# Define the custom column for the state label
odor_stimulus.add_column(name='stimulus', 
                        description='Odor Stim')

# Add the TimeIntervals table to the NWBFile
nwbfile.add_time_intervals(odor_stimulus)

# --- Add ODORS ---
odor_data = odor_1_times  # Your input variable
state_label = 'Odor 1'

for row in odor_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    odor_stimulus.add_row(
        start_time=start_time,
        stop_time=stop_time,
        stimulus=state_label
    )

print(f"Successfully added {len(odor_data)} '{state_label}' intervals.")

odor_data = odor_2_times  # Your input variable
state_label = 'Odor 2'

for row in odor_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    odor_stimulus.add_row(
        start_time=start_time,
        stop_time=stop_time,
        stimulus=state_label
    )

print(f"Successfully added {len(odor_data)} '{state_label}' intervals.")

odor_data = odor_3_times  # Your input variable
state_label = 'Odor 3'

for row in odor_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    odor_stimulus.add_row(
        start_time=start_time,
        stop_time=stop_time,
        stimulus=state_label
    )

print(f"Successfully added {len(odor_data)} '{state_label}' intervals.")

odor_data = odor_empty_times  # Your input variable
state_label = 'Odor Empty'

for row in odor_data:
    start_time = row[0]
    stop_time = row[1]
    
    # Add the row to the TimeIntervals table
    odor_stimulus.add_row(
        start_time=start_time,
        stop_time=stop_time,
        stimulus=state_label
    )

print(f"Successfully added {len(odor_data)} '{state_label}' intervals.")


import h5py

# "TES_sResp_M03_20240621","TES_sResp_M03_20240622","TES_sResp_M03_20240623",
#                            "TES_sResp_M05_20240729","TES_sResp_M05_20240730","TES_sResp_M05_20240731"]

names_w_tracking_ripples_maze_change = ['TES_sResp_M01_20240318','TES_sResp_M02_20240318','TES_sResp_M03_20240624']

path = 'TES_sResp_M03_20240624.nwb'

# Open the file using h5py directly
with h5py.File(path, 'r+') as f:
    # Navigate to the ElectrodeGroup1 location
    # The path in NWB is always: general/extracellular_ephys/GROUP_NAME
    target_path = 'general/extracellular_ephys/ElectrodeGroup1'
    
    if target_path in f:
        group = f[target_path]
        
        # In NWB, 'description' is stored as an Attribute
        new_desc = '60 degree insertion targetting hippocampus and neocortex'
        
        # Overwrite the attribute
        group.attrs['description'] = new_desc
        
        print(f"Successfully updated description in HDF5 for {path}")
    else:
        print(f"Could not find {target_path} in the file.")


from pynwb import NWBHDF5IO

path = 'TES_sResp_M03_20240624.nwb'

# Open in read-only mode to verify the disk content
with NWBHDF5IO(path, mode='r') as io:
    nwbfile = io.read()
    
    if 'ElectrodeGroup1' in nwbfile.electrode_groups:
        
        eg = nwbfile.electrode_groups['ElectrodeGroup1']
        
        print("--- Verification ---")
        print(f"Name:        {eg.name}")
        print(f"Description: {eg.description}")
        print(f"Location:    {eg.location}")
    else:
        print("Error: ElectrodeGroup1 not found.")